# 03 · Анализ

Числа и таблицы по результатам прогона — без DOLFINx, только файлы: работает и на
ноутбуке с копией папки `runs/<имя>/`.

1. Какие прогоны есть и что в них посчитано (манифест `run.json`).
2. Временные ряды (`series.csv`) и механические биомаркеры.
3. Карты активации (`activation.npz`): время активации, скорость проведения, APD по регионам,
   блок проведения, реституция.
4. Снимки полей: статистика по регионам.
5. Серия: сводная таблица «параметры → биомаркеры».
6. Экспорт таблиц в `runs/<имя>/analysis/`.

Графики — в `04_visualization.ipynb`.

In [ ]:
# Общая настройка: пакет cardiac_em и помощники ноутбуков доступны из любой папки проекта
import sys
from pathlib import Path

for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "notebooks" / "nbtools.py").exists():
        sys.path.insert(0, str(_p / "notebooks"))
        break
from nbtools import ROOT, RUNS, run_stream, style  # noqa: E402

print("корень проекта:", ROOT)

In [ ]:
import json
import numpy as np
import pandas as pd
from cardiac_em.analysis import biomarkers as bm
from cardiac_em.analysis import open_run, open_sweep
from nbtools import list_runs

pd.set_option("display.precision", 4)
pd.DataFrame(list_runs())

## 1. Прогон

Имя папки в `runs/` (или полный путь к папке с `run.json`).

In [ ]:
RUN_DIR = RUNS / "nb_quick"          # ночная серия: RUNS / "night" / "run_000"

run = open_run(RUN_DIR)
m = run.manifest
print(f"статус: {run.status},  счёт {m.get('elapsed_s')} с,  "
      f"t = {m['schedule']['t_start_ms']:g} → {m['schedule']['t_end_ms']:g} мс")
print(f"модель клетки: {m['models']['cell']},  материал: {m['models']['material']}")
print(f"перенос T_act: {m['models']['transfer']}")
for part in ("electric", "mechanical"):
    g = run.meshes[part]
    print(f"сетка {part:10}: {g['nx']}×{g['ny']} на {g['lx_mm']}×{g['ly_mm']} мм, {g['n_cells']} ячеек")
env = m.get("environment", {})
print(f"окружение: DOLFINx {env.get('dolfinx')}, рангов MPI {env.get('mpi_size')}, "
      f"git {env.get('git', {}).get('commit', '—')[:10]}")
for w in run.warnings:
    print("  [!]", w)

### Параметры прогона

In [ ]:
cfg = run.config
tb = cfg["tissue_base"]
print("T_MAX =", tb["active"]["t_max"], "кПа;  D =", tb["conduction"]["d_long"], "/",
      tb["conduction"]["d_trans"], "мм²/мс;  волокна", tb["conduction"]["fiber_angle_deg"], "°")
print("параметры клетки для всей ткани:", cfg.get("cell_params") or "по умолчанию")
print("стимулы, мс:", cfg["stimulus"]["times_ms"])
regions = pd.DataFrame([{"№": i + 1, "имя": r.get("name"), "форма": r["shape"],
                         **r.get("params", {}), **r.get("overrides", {})}
                        for i, r in enumerate(cfg.get("regions", []))])
regions if not regions.empty else print("регионов нет — однородная ткань")

## 2. Временные ряды и механика

`series.csv` — по строке на механический шаг. `t_act_mech_*` — сила, переданная клетками
(у TNNPM — при нулевой скорости волокна), `t_act_actual_*` — действующее напряжение с учётом
сила–скорость.

In [ ]:
series = run.series(as_frame=True)
series.describe().T[["min", "max", "mean"]]

In [ ]:
mech = bm.mechanics_summary(run.series())
pd.Series(mech, name="механика").to_frame()

## 3. Карты активации

По каждому узлу электрической сетки и каждому удару: время активации (пересечение порога
модели снизу вверх), реполяризации (APD на уровне `apd_level`, по умолчанию 90 %) и пик.

In [ ]:
try:
    maps = run.activation()
except FileNotFoundError as exc:
    maps = None
    print(exc)
else:
    print(f"ударов: {maps.n_beats},  узлов: {len(maps.coords)},  порог {maps.threshold:g},  "
          f"APD{int(round(maps.apd_level * 100))}")

In [ ]:
def beat_table(maps, window=None):
    rows = []
    for k in range(maps.n_beats):
        act, apd = maps.act[:, k], maps.apd[:, k]
        s = bm.activation_summary(act)
        a = bm.apd_summary(apd)
        rows.append({"удар": k + 1, **s,
                     "CV_x, мм/мс": bm.conduction_velocity(maps.coords, act, axis=0, window=window),
                     "APD mean, мс": a["mean_ms"], "APD min": a["min_ms"], "APD max": a["max_ms"],
                     "дисперсия APD": a["dispersion_ms"]})
    return pd.DataFrame(rows)

CV_WINDOW = None      # (x_min, x_max) мм — окно для скорости; None — средние 40 % области
beats = beat_table(maps, CV_WINDOW) if maps is not None else pd.DataFrame()
beats

Локальная скорость проведения |∇t_act|⁻¹ (медиана по области — устойчивее, чем среднее:
в зоне стимула градиент почти нулевой) и APD по регионам (0 — базовая ткань, *i* — регион *i*
конфигурации).

In [ ]:
if maps is not None and maps.n_beats:
    g = run.meshes["electric"]
    hx, hy = g["lx_mm"] / g["nx"], g["ly_mm"] / g["ny"]
    BEAT = -1                         # удар для CV и APD: 0 — первый, -1 — последний
    b = BEAT % maps.n_beats
    cv = bm.cv_field(maps.grid(maps.act[:, b]), hx, hy)
    print(f"локальная CV (удар {b + 1}): медиана {np.nanmedian(cv):.3f} мм/мс, "
          f"5–95 %: {np.nanpercentile(cv, 5):.3f}–{np.nanpercentile(cv, 95):.3f}")
    names = {0: "базовая ткань", **{i + 1: r.get("name") or f"регион {i + 1}"
                                    for i, r in enumerate(cfg.get("regions", []))}}
    by_region = pd.DataFrame(bm.apd_summary(maps.apd[:, b], maps.region)["by_region"]).T
    by_region.index = [names.get(i, i) for i in by_region.index]
    display(by_region.rename(columns={"mean": "APD mean, мс", "std": "std", "min": "min", "max": "max", "n": "узлов"}))

In [ ]:
if maps is not None and maps.n_beats > 1:
    for k in range(maps.n_beats - 1):
        blk = bm.conduction_block(maps.act, k)
        print(f"удар {k + 1} → {k + 2}: не проведено в {blk['n_blocked']} узлах "
              f"({100 * blk['fraction_blocked']:.1f} %)")
    rest = bm.restitution(maps.act, maps.repol)
    print(f"пар (DI, APD) для реституции: {len(rest['di_ms'])}")
elif maps is not None:
    print("один удар — блок проведения и реституция требуют нескольких")

## 4. Снимки: статистика по регионам

Для каждого снимка — средние по регионам: потенциал на электрической сетке, растяжение
волокна и действующее напряжение на механической.

In [ ]:
V_NAME = "V" if run.manifest["models"]["cell"].startswith("tnnpm") else run.manifest["models"]["cell_states"][0]
rows = []
for snap in run.snapshots():
    for r in np.unique(snap["m_region"]):
        me, mm = snap["e_region"] == r, snap["m_region"] == r
        rows.append({"t, мс": snap.t_ms, "регион": int(r),
                     f"{V_NAME} средн.": snap.state(V_NAME)[me].mean(),
                     "λ_f средн.": snap["m_lambda_f"][mm].mean(),
                     "λ_f мин.": snap["m_lambda_f"][mm].min(),
                     "T_act средн., кПа": snap["m_t_act"][mm].mean()})
snap_table = pd.DataFrame(rows)
snap_table if not snap_table.empty else print("снимков нет (output.snapshot_times_ms)")

## 5. Серия

Сводная таблица по серии: по строке на точку — её параметры и биомаркеры. Функция `metrics`
определяет, какие числа нужны; её можно менять как угодно.

In [ ]:
SWEEP_DIR = RUNS / "nb_quick_sweep"  # ночная серия: RUNS / "night"
SWEEP_BEAT = -1                      # удар для метрик серии: -1 — последний (установившийся)

def metrics(r):
    out = bm.mechanics_summary(r.series())
    try:
        mp = r.activation()
        b = SWEEP_BEAT % mp.n_beats
        out.update(bm.activation_summary(mp.act[:, b]))
        a = bm.apd_summary(mp.apd[:, b], mp.region)
        out["APD mean, мс"] = a["mean_ms"]
        for reg, st in a["by_region"].items():
            out[f"APD регион {reg}"] = st["mean"]
    except FileNotFoundError:
        pass
    return out

if (SWEEP_DIR / "sweep.json").exists():
    sweep = open_sweep(SWEEP_DIR)
    sweep_table = pd.DataFrame(sweep.table(metrics))
    done = sum(p["status"] == "finished" for p in sweep.points)
    print(f"посчитано точек: {done} из {len(sweep.points)}")
    if sweep_table.empty:
        sweep_table = None
        print("запустите серию в 01_run.ipynb (раздел 6, RUN_SWEEP или RUN_SWEEP_MPI)")
    else:
        display(sweep_table)
else:
    sweep_table = None
    print(f"серии в {SWEEP_DIR.relative_to(ROOT)} нет — запустите её в 01_run.ipynb (раздел 6)")

## 6. Экспорт

Таблицы — в `runs/<имя>/analysis/` (CSV открываются в Excel/Origin), сводка — в JSON.

In [ ]:
out_dir = run.dir / "analysis"
out_dir.mkdir(exist_ok=True)
series.to_csv(out_dir / "series.csv", index=False)
if not beats.empty:
    beats.to_csv(out_dir / "beats.csv", index=False)
if not snap_table.empty:
    snap_table.to_csv(out_dir / "snapshots_by_region.csv", index=False)
if sweep_table is not None:
    sweep_table.to_csv(SWEEP_DIR / "sweep_table.csv", index=False)

summary = {"run": str(run.dir.relative_to(ROOT)), "mechanics": mech,
           "beats": beats.to_dict(orient="records")}
(out_dir / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2, default=float),
                                      encoding="utf-8")
print("записано в", out_dir.relative_to(ROOT), ":", sorted(p.name for p in out_dir.iterdir()))